### 1. Load notes
  
    Read all .md files from the notes folder. We try to preserve the metadata along the contents.
    The resulting object will look like:
```python
documents = [
    {
        "content": "...text from note 1...",
        "metadata": {"source": "notes/foo.md"},
    },
    {
        "content": "...text from note 2...",
        "metadata": {"source": "notes/bar.md"},
    },
]
```

In [ ]:
from pathlib import Path

# Load all notes
def load_notes(notes_dir):
    notes = []
    for note_file in Path(notes_dir).rglob('*.md'):
        with open(note_file, 'r', encoding='utf-8') as f:
            notes.append({
                'content': f.read(),
                'metadata': {'source': str(note_file)}
            })
    return notes

notes = load_notes('notes')
print(f'Loaded {len(notes)} notes')
for note in notes:
    print(f"{note['metadata']['source']}: {len(note['content'])} chars")

Loaded 3 notes
notes/Notes-ansible.md: 7784 chars
notes/example-rag.md: 960 chars
notes/Notes-cypress.md: 2951 chars


### 2. Chunking
   
   We split each file into overlapping chunks. Each chunk looks like this:

```python
{
    "content": "...chunk text...",
    "metadata": {
        "source": "notes/Notes-ansible.md",
        "chunk": 0,
        "start": 0,
        "end": 800
    }
}
```

In [8]:
def chunk_notes(notes, chunk_size=800, chunk_overlap=150):
    chunks = []
    chunk_size = 800
    chunk_overlap = 150
    for note in notes:
        content = note['content']
        print(f'Processing {note["metadata"]["source"]} with {len(content)} chars')
        content = note['content']
        chunk_id = 0
        step = chunk_size - chunk_overlap
        i = 0
        while i < len(content):
            end = min(i + chunk_size, len(content))
            chunk = content[i:end]
            chunks.append({
                'content': chunk,
                'metadata': {
                    'source': note['metadata']['source'],
                    'chunk_id': chunk_id,
                    'start': i,
                    'end': end
                }
            })
            chunk_id += 1
            print(f'\tchunk {chunk_id}, chars {i}-{end}')
            i += step
    return chunks

chunks = chunk_notes(notes)
print(f'Created {len(chunks)} chunks')

Processing notes/Notes-ansible.md with 7784 chars
	chunk 1, chars 0-800
	chunk 2, chars 650-1450
	chunk 3, chars 1300-2100
	chunk 4, chars 1950-2750
	chunk 5, chars 2600-3400
	chunk 6, chars 3250-4050
	chunk 7, chars 3900-4700
	chunk 8, chars 4550-5350
	chunk 9, chars 5200-6000
	chunk 10, chars 5850-6650
	chunk 11, chars 6500-7300
	chunk 12, chars 7150-7784
Processing notes/example-rag.md with 960 chars
	chunk 1, chars 0-800
	chunk 2, chars 650-960
Processing notes/Notes-cypress.md with 2951 chars
	chunk 1, chars 0-800
	chunk 2, chars 650-1450
	chunk 3, chars 1300-2100
	chunk 4, chars 1950-2750
	chunk 5, chars 2600-2951
Created 19 chunks


### 3. Embedding
   
   Each chunk gets converted from text into a numeric vector, which allows us to do semantic search.

In [1]:
from langchain_huggingface.embeddings import HuggingFaceEndpointEmbeddings
from dotenv import load_dotenv

load_dotenv()

embeddings = HuggingFaceEndpointEmbeddings(
    model="sentence-transformers/all-MiniLM-L6-v2",
    task="feature-extraction",
)

/Users/diaa/git/langchain/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def embed_chunks(chunks, embeddings):
    texts = [chunk['content'] for chunk in chunks]
    return embeddings.embed_documents(texts)
vectors = embed_chunks(chunks, embeddings)
print(f'Generated {len(vectors)} vectors of dimension {len(vectors[0])}')

### Manual Similarity Search
For performing similarity search for a question, we embed the question, compare the question vector to each chunk vector, and finally find the most similar vector.

In [ ]:
import math

def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    magnitude_a = math.sqrt(sum(x * x for x in a))
    magnitude_b = math.sqrt(sum(y * y for y in b))
    return dot_product / (magnitude_a * magnitude_b)

question = "How to run an Ansible playbook?"
question_vector = embeddings.embed_query(question)
similarities = [{'idx': i, 'similarity': cosine_similarity(question_vector, vector)} for i, vector in enumerate(vectors)]
similarities.sort(key=lambda x: x['similarity'], reverse=True)
top_results = similarities[:3]
for sim in top_results:
    idx = sim['idx']
    print(f"{chunks[idx]['metadata']['source']} (chunk {chunks[idx]['metadata']['chunk_id']}): {sim['similarity']:.4f}")
    # print(chunks[idx]['content'][:200] + '...\n')

notes/Notes-ansible.md (chunk 5): 0.6985
notes/Notes-ansible.md (chunk 4): 0.6967
notes/Notes-ansible.md (chunk 0): 0.6953


### Generation
In this step we send the question and the top retrieved chunks to the LLM with a prompt like:

```
Answer the question using only the context below.
If the context does not contain the answer, say you do not know.

Question:
...

Context:
[1] source/chunk info
chunk text

[2] source/chunk info
chunk text

Answer:
```

In [ ]:
# from langchain_huggingface import HuggingFaceEndpoint
# llm = HuggingFaceEndpoint(
#     model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
#     task="",
#     provider="auto",
#     max_new_tokens=300,
#     temperature=0.2,
# )

def build_prompt(question, top_results, chunks):
    context_parts = []
    for i, result in enumerate(top_results, start=1):
        idx = result['idx']
        context_parts.append(
            f'[{i}] Source: {chunks[idx]["metadata"]["source"]}, chunk {chunks[idx]["metadata"]["chunk_id"]}\n'
            f'{chunks[idx]["content"]}'
        )
    context = "\n\n".join(context_parts)
    prompt = f"""
    Answer the question using only the context below.
    If the context does not contain the answer, say you do not know.

    Question:
    {question}

    Context:
    {context}

    Answer:
    """
    return prompt
# answer = llm.invoke(prompt)
# print(answer)

from huggingface_hub import InferenceClient
import os
load_dotenv()

def ask_llm(prompt) -> str|None:
    client = InferenceClient(
        model="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
        provider="featherless-ai",
        api_key=os.environ["HUGGINGFACEHUB_API_TOKEN"],
    )
    response = client.chat_completion(
        messages=[
            {"role": "user", "content": prompt}
        ],
        max_tokens=300,
        temperature=0.2,
    )
    return response.choices[0].message.content

In [ ]:
prompt = build_prompt(question, top_results, chunks)
answer = ask_llm(prompt)
print(answer)

## Using Chroma
Chroma handles many things that we did manually:
- chunks -> Chroma collection
- question -> Chroma similarity_search
- top chunks -> prompt -> HF chat model

### Converting chunks to Langchain Documents:

In [ ]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content=chunk['content'],
        metadata=chunk['metadata']
    ) for chunk in chunks
]
print(f"Created {len(documents)} documents")
print(documents[0].metadata)
print(documents[0].page_content[:300])

### Using Chroma Vector

In [ ]:
from langchain_chroma import Chroma

vector_store = Chroma(
        collection_name="notes",
        embedding_function=embeddings,
        persist_directory="./chroma_db",
)
vector_store.add_documents(documents)
print("Added documents to Chroma DB")

Added documents to Chroma DB


### Using Chroma Similarity Search

In [ ]:
results = vector_store.similarity_search(question, k=3)
for i, doc in enumerate(results, start=1):
    print(f"[{i}] Source: {doc.metadata['source']}, chunk {doc.metadata['chunk_id']}")

### Reloading Vector Store

In [4]:
from langchain_chroma import Chroma

vector_store = Chroma(
    collection_name="notes",
    persist_directory="./chroma_db",
    embedding_function=embeddings,
)
question = "How to run an Ansible playbook?"
results = vector_store.similarity_search(question, k=3)
for i, doc in enumerate(results, start=1):
    print(f"[{i}] Source: {doc.metadata['source']}, chunk {doc.metadata['chunk_id']}")

[1] Source: notes/Notes-ansible.md, chunk 5
[2] Source: notes/Notes-ansible.md, chunk 4
[3] Source: notes/Notes-ansible.md, chunk 0


### Full Query Phase
question -> Chroma retrieval -> context -> LLM answer

In [ ]:
def retrieve_context(question: str, vector_store: Chroma, k=3):
    return vector_store.similarity_search(question, k=k)

def format_context(docs):
    context_parts = []
    for i, doc in enumerate(docs, start=1):
        source = doc.metadata["source"]
        chunk_id = doc.metadata["chunk_id"]

        context_parts.append(
            f"[{i}] Source: {source}, chunk {chunk_id}\n"
            f"{doc.page_content}"
        )
    return "\n\n".join(context_parts)

from textwrap import dedent

def answer_question(question: str, vector_store: Chroma, k: int = 3):
    docs = retrieve_context(question, vector_store, k=k)
    context = format_context(docs)

    prompt = dedent(f"""
        Answer the question using only the context below.
        If the context does not contain the answer, say you do not know.

        Question:
        {question}

        Context:
        {context}

        Answer:
    """).strip()

    answer = ask_llm(prompt)
    return answer, docs

# Module Test

In [ ]:
import rag
from pathlib import Path
rag.index_notes(Path('notes'))

In [ ]:
import rag
embeddings = rag.create_embeddings()
vector_store = rag.load_vector_store(embeddings)
llm = rag.create_llm()

question = "How to run an Ansible playbook?"
answer, docs, sources = rag.answer_question(question, vector_store, llm)

/Users/diaa/git/langchain/venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
print(answer)

To run an Ansible playbook, you need to specify the `playbook.yml` file and the target hosts or groups. Here's an example:

```yaml
---
  # Play
  - name: install essential packages
    # Target host. Can be `all`, a single host, or a group
    hosts: linux
    # Become the sudo user
    become: yes
    vars:
      # SSH credentials if not specified in `hosts` or command
      ansible_user: myuser
      ansible_password: mypassword
    # List of tasks to run
    tasks:
      - name: ensure nano is installed
        # Module
        apt:
            name: nano
            state: latest
        # OR short form
        apt: name=nano state=latest
      # It is possible to run commands directly.
      # Not recommended as it's not item potent (see introduction for meaning).
      - command: apt install -y nano

```

In this example, the playbook is named `playbook.yml` and it targets the `linux` group. You can also specify other hosts or groups using the `hosts` variable.

To run the playb